# Capítulo 1 — Tensors and Shapes

Siguiendo [dmol.pub/math/tensors-and-shapes.html](https://dmol.pub/math/tensors-and-shapes.html).

Este es el capítulo de repaso matemático, y es el único del libro que **no va de química ni de ML**:
va de manejar arrays de numpy. Puede parecer que sobra. No sobra.

La razón es esta: en ML **todo** es un array multidimensional, y prácticamente todos los bugs que
vas a tener son **errores de forma** — multiplicar un `(32, 17)` por un `(17, 32)` cuando querías
otra cosa, o que numpy "encaje" silenciosamente dos arrays que no debía y te devuelva un resultado
del tamaño equivocado sin dar error.

Los dos síntomas clásicos:

- `ValueError: operands could not be broadcast together` → molesto pero honesto, te avisa.
- Un resultado con la forma equivocada y **ningún error** → este es el que te come una tarde.

Todo el capítulo es numpy puro:

In [1]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)   # imprimir sin notación científica
print("numpy", np.__version__)

numpy 2.4.6


---

## 1.0 El vocabulario: tensor, rank y shape

Un **tensor** es, para lo que nos ocupa, simplemente **un array de números con $n$ dimensiones**.
El nombre viene de la física y en ML se usa de forma más laxa: aquí "tensor" = `np.array`.

Dos palabras que hay que tener clavadas:

| Término | Qué es | Cómo se pregunta |
|---|---|---|
| **rank** (rango) | **cuántas** dimensiones tiene | `a.ndim` |
| **shape** (forma) | **cuánto mide** cada dimensión | `a.shape` |

Y la escalera de rangos, con lo que significa cada uno en química:

| Rank | Nombre | Ejemplo en química | Shape |
|---|---|---|---|
| 0 | escalar | la solubilidad de **una** molécula | `()` |
| 1 | vector | los 17 descriptores de **una** molécula | `(17,)` |
| 2 | matriz | los descriptores de **todas** las moléculas | `(9982, 17)` |
| 3 | tensor | las coordenadas XYZ de un lote de moléculas | `(32, 50, 3)` |
| 4 | tensor | un lote de imágenes en color | `(32, 128, 128, 3)` |

Esa última fila es la razón de que se hable de "tensores" y no de "matrices": en cuanto pasas de
rank 2, el álgebra de matrices del instituto ya no te sirve para escribir lo que quieres hacer.

**La convención de la primera dimensión.** Casi siempre, el **primer eje es el batch** — los
ejemplos independientes. Un `(32, 50, 3)` se lee "32 moléculas, cada una con 50 átomos, cada átomo
con 3 coordenadas". Acostúmbrate a leer las formas de izquierda a derecha así, en voz alta: te
ahorra la mitad de los errores.

In [2]:
escalar = np.array(3.7)
vector  = np.array([1.0, 2.0, 3.0])
matriz  = np.array([[1.0, 2.0, 3.0],
                    [4.0, 5.0, 6.0]])
tensor3 = np.zeros((2, 4, 3))       # 2 moléculas x 4 átomos x 3 coordenadas

for nombre, t in [("escalar", escalar), ("vector", vector), ("matriz", matriz), ("tensor3", tensor3)]:
    print(f"{nombre:9s} rank = {t.ndim}   shape = {str(t.shape):12s} nº de números = {t.size}")

escalar   rank = 0   shape = ()           nº de números = 1
vector    rank = 1   shape = (3,)         nº de números = 3
matriz    rank = 2   shape = (2, 3)       nº de números = 6
tensor3   rank = 3   shape = (2, 4, 3)    nº de números = 24


Fíjate en la diferencia entre `()` , `(3,)` y `(1, 3)`:

- `()` es un escalar: un número suelto.
- `(3,)` es un vector de 3. La coma no es un adorno — `(3,)` es una tupla de un elemento; sin ella,
  `(3)` sería simplemente el número 3 entre paréntesis.
- `(1, 3)` es una **matriz** de 1 fila y 3 columnas.

Los tres contienen "casi lo mismo" y numpy los trata distinto. Esta confusión es la fuente número
uno de errores de forma, y por eso la sección 1.4 va entera de convertir unos en otros.

---

## 1.1 Notación de Einstein y `einsum`

Antes de operar, hace falta una manera de **escribir** lo que quieres hacer. La notación matricial
clásica se queda corta en cuanto hay más de dos índices, así que se usa la **notación de Einstein**.

La regla, que es una sola:

> Escribes los índices de las entradas y los de la salida. **Todo índice que aparece en la entrada
> pero no en la salida, se suma.**

Ejemplos, con $A$ una matriz de índices $i$ (filas) y $j$ (columnas):

| Notación | Qué hace | En fórmula |
|---|---|---|
| `ij->` | suma todo | $\sum_i \sum_j A_{ij}$ |
| `ij->i` | suma cada fila (desaparece $j$) | $\sum_j A_{ij}$ |
| `ij->j` | suma cada columna (desaparece $i$) | $\sum_i A_{ij}$ |
| `ij->ji` | transpone (no suma nada) | $A_{ji}$ |
| `i,i->` | producto escalar | $\sum_i v_i u_i$ |
| `ij,j->i` | matriz por vector | $\sum_j A_{ij} v_j$ |
| `ij,jk->ik` | matriz por matriz | $\sum_j A_{ij} B_{jk}$ |

Y numpy lo implementa **literalmente** con `np.einsum`, escribiendo la misma cadena:

In [3]:
A = np.array([[1, 2],
              [3, 4]])
v = np.array([10, 20])

print("A =\n", A, "\nv =", v, "\n")

print("einsum('ij->',   A)     =", np.einsum("ij->", A),   "  <- suma de todo: 1+2+3+4")
print("einsum('ij->i',  A)     =", np.einsum("ij->i", A),  "  <- por filas:    [1+2, 3+4]")
print("einsum('ij->j',  A)     =", np.einsum("ij->j", A),  "  <- por columnas: [1+3, 2+4]")
print("einsum('ij->ji', A)     =\n", np.einsum("ij->ji", A), "  <- transpuesta (nada se suma)")
print()
print("einsum('i,i->',    v, v) =", np.einsum("i,i->", v, v),     "  <- producto escalar: 10*10 + 20*20")
print("einsum('ij,j->i',  A, v) =", np.einsum("ij,j->i", A, v),   "  <- matriz por vector")
print("einsum('ij,jk->ik',A, A) =\n", np.einsum("ij,jk->ik", A, A), " <- matriz por matriz")

A =
 [[1 2]
 [3 4]] 
v = [10 20] 

einsum('ij->',   A)     = 10   <- suma de todo: 1+2+3+4
einsum('ij->i',  A)     = [3 7]   <- por filas:    [1+2, 3+4]
einsum('ij->j',  A)     = [4 6]   <- por columnas: [1+3, 2+4]
einsum('ij->ji', A)     =
 [[1 3]
 [2 4]]   <- transpuesta (nada se suma)

einsum('i,i->',    v, v) = 500   <- producto escalar: 10*10 + 20*20
einsum('ij,j->i',  A, v) = [ 50 110]   <- matriz por vector
einsum('ij,jk->ik',A, A) =
 [[ 7 10]
 [15 22]]  <- matriz por matriz


**¿Por qué molestarse, si existe `@` para multiplicar matrices?**

Por tres razones, y la tercera es la de verdad:

1. **Se lee.** `np.einsum("ij,jk->ik", A, B)` dice qué índice se contrae. `A @ B` te obliga a
   recordar la convención.
2. **Hace cosas que `@` no puede.** Contraer solo el último eje de un rank-4 con el primero de un
   rank-3 no tiene símbolo; en einsum es una cadena.
3. **Es la notación de los papers.** Cuando llegues a los capítulos de redes equivariantes (10) y de
   attention (12), las ecuaciones vienen escritas exactamente así. El libro usa einsum precisamente
   para que la ecuación del paper y el código sean la misma línea.

Ejemplo real que verás en el capítulo 12 (*attention*): calcular el producto escalar de cada query
con cada key, para un lote entero, es

```python
np.einsum("bqd,bkd->bqk", Q, K)
```

*(batch, query, dimensión) × (batch, key, dimensión) → (batch, query, key)*: la `d` desaparece, así
que se suma sobre ella; la `b` está en los tres sitios, así que se hace por separado para cada
elemento del lote. Una línea, y se entiende sin ejecutarla.

---

## 1.2 Operaciones: qué le pasa a la forma

Las operaciones se dividen en dos familias, y lo que las distingue es **qué le hacen a la forma**.

### Elemento a elemento (*element-wise*): la forma no cambia

`+`, `-`, `*`, `/`, `**`, `np.exp`, `np.sqrt`... se aplican número a número y devuelven algo de la
**misma forma**.

Cuidado con una trampa de notación: en numpy `*` es **multiplicación elemento a elemento**, NO
producto matricial. El producto matricial es `@` (o `np.matmul`, o einsum). Es un error clásico al
venir de MATLAB.

### Reducciones: la forma encoge

`np.sum`, `np.mean`, `np.max`, `np.min`, `np.any`, `np.all`... **eliminan** uno o varios ejes.

El parámetro clave es `axis`, y la manera correcta de leerlo es:

> `axis=k` significa **"el eje k desaparece"**, colapsado por la operación.

Con un tensor de forma `(2, 3, 4)`:

| Operación | Eje que desaparece | Forma resultante |
|---|---|---|
| `np.sum(B)` | todos | `()` |
| `np.sum(B, axis=0)` | el 0 (el de tamaño 2) | `(3, 4)` |
| `np.sum(B, axis=1)` | el 1 | `(2, 4)` |
| `np.sum(B, axis=(1, 2))` | el 1 y el 2 | `(2,)` |
| `np.sum(B, axis=2, keepdims=True)` | el 2, pero deja un hueco de tamaño 1 | `(2, 3, 1)` |

In [4]:
B = np.arange(24).reshape(2, 3, 4)
print("B.shape =", B.shape, "  rank =", B.ndim, "  nº de números =", B.size)
print()

print("np.sum(B)                       -> ", np.sum(B), "  (un escalar)")
print("np.sum(B, axis=0).shape         -> ", np.sum(B, axis=0).shape)
print("np.sum(B, axis=1).shape         -> ", np.sum(B, axis=1).shape)
print("np.sum(B, axis=(1,2))           -> ", np.sum(B, axis=(1, 2)), " shape", np.sum(B, axis=(1, 2)).shape)
print("np.sum(B, axis=2, keepdims=True).shape ->", np.sum(B, axis=2, keepdims=True).shape)
print()
print("(B * 2).shape  ->", (B * 2).shape, "  <- element-wise: la forma NO cambia")
print("(B + B).shape  ->", (B + B).shape)

B.shape = (2, 3, 4)   rank = 3   nº de números = 24

np.sum(B)                       ->  276   (un escalar)
np.sum(B, axis=0).shape         ->  (3, 4)
np.sum(B, axis=1).shape         ->  (2, 4)
np.sum(B, axis=(1,2))           ->  [ 66 210]  shape (2,)
np.sum(B, axis=2, keepdims=True).shape -> (2, 3, 1)

(B * 2).shape  -> (2, 3, 4)   <- element-wise: la forma NO cambia
(B + B).shape  -> (2, 3, 4)


**`keepdims=True` merece un párrafo propio** porque parece un capricho y no lo es.

Compara: sumar el eje 2 de un `(2, 3, 4)` da `(2, 3)` normalmente, y `(2, 3, 1)` con `keepdims`.
El segundo conserva el "hueco" del eje eliminado.

¿Para qué? Porque casi siempre quieres **volver a combinar** el resultado con el original — por
ejemplo para normalizar: dividir cada elemento por la suma de su fila. Con `keepdims=True` la
división funciona sola gracias al broadcasting (siguiente sección); sin él, tienes que ir
recolocando ejes a mano. Es el caso de uso del 90 % de los `keepdims` que verás.

In [5]:
# normalizar cada fila para que sume 1 — el caso de uso tipico de keepdims
M = np.array([[1.0, 1.0, 2.0],
              [2.0, 3.0, 5.0]])

sumas = M.sum(axis=1, keepdims=True)      # (2, 1) en vez de (2,)
print("M.shape     ", M.shape)
print("sumas.shape ", sumas.shape, "->\n", sumas)
print("\nM / sumas =\n", M / sumas)
print("\ncomprueba, cada fila suma:", (M / sumas).sum(axis=1))

M.shape      (2, 3)
sumas.shape  (2, 1) ->
 [[ 4.]
 [10.]]

M / sumas =
 [[0.25 0.25 0.5 ]
 [0.2  0.3  0.5 ]]

comprueba, cada fila suma: [1. 1.]


---

## 1.3 Broadcasting

Esta es **la sección importante del capítulo**. El broadcasting es la regla por la que numpy te deja
operar arrays de formas distintas, estirando automáticamente el pequeño para que encaje con el grande.

Es lo que hace que el código de ML sea corto y legible. También es lo que hace que, cuando te
equivocas, el error sea silencioso.

### La regla

numpy alinea las formas **por el final** (de derecha a izquierda) y compara eje por eje. Dos ejes
son compatibles si:

1. **son iguales**, o
2. **uno de los dos vale 1** (ese se "estira" repitiendo su contenido), o
3. **uno de los dos no existe** (se considera de tamaño 1).

Si algún par no cumple ninguna, salta `ValueError`.

Ejemplo, lo que hicimos en el capítulo 2 para estandarizar:

```
X        (9982, 17)
medias   (      17,)      <- se alinea por el final
--------------------
resultado(9982, 17)       <- el vector de 17 medias se aplica a las 9982 filas
```

Y uno que falla:

```
X        (9982, 17)
otro     (9982,    )      <- se alinea por el final: 17 contra 9982
--------------------
ValueError
```

Es contraintuitivo: el que parece "obvio" (un valor por molécula) es justo el que no funciona,
porque numpy alinea **por el final**, no por el principio. La solución es `X - otro[:, np.newaxis]`,
que lo convierte en `(9982, 1)` y entonces sí alinea. Eso es la sección 1.4.

In [6]:
X = np.arange(12).reshape(4, 3).astype(float)   # 4 "moléculas" x 3 "features"
print("X =\n", X, "\n")

por_columna = np.array([100., 200., 300.])      # (3,) -> un valor por FEATURE
print("X - por_columna  ->  (4,3) con (3,)  = FUNCIONA\n", X - por_columna, "\n")

por_fila = np.array([1., 2., 3., 4.])           # (4,) -> un valor por MOLÉCULA
try:
    X - por_fila
except ValueError as e:
    print("X - por_fila   ->  (4,3) con (4,)  = FALLA")
    print("   ", e)

print("\nX - por_fila[:, np.newaxis]  ->  (4,3) con (4,1)  = FUNCIONA\n",
      X - por_fila[:, np.newaxis])

X =
 [[ 0.  1.  2.]
 [ 3.  4.  5.]
 [ 6.  7.  8.]
 [ 9. 10. 11.]] 

X - por_columna  ->  (4,3) con (3,)  = FUNCIONA
 [[-100. -199. -298.]
 [ -97. -196. -295.]
 [ -94. -193. -292.]
 [ -91. -190. -289.]] 

X - por_fila   ->  (4,3) con (4,)  = FALLA
    operands could not be broadcast together with shapes (4,3) (4,) 

X - por_fila[:, np.newaxis]  ->  (4,3) con (4,1)  = FUNCIONA
 [[-1.  0.  1.]
 [ 1.  2.  3.]
 [ 3.  4.  5.]
 [ 5.  6.  7.]]


### El ejemplo que de verdad importa: la matriz de distancias

Este patrón aparece en todos los capítulos de moléculas del libro (8, 9, 10, 16, 17), así que vale
la pena entenderlo bien.

**Problema:** tienes las coordenadas de $N$ átomos, un array `(N, 3)`. Quieres la matriz `(N, N)`
con la distancia entre cada par.

La solución ingenua es un doble bucle. La solución real es broadcasting:

```python
diferencias = coords[:, None, :] - coords[None, :, :]
```

Paso a paso, que es donde está la magia:

```
coords[:, None, :]   ->  (N, 1, 3)     "cada átomo como fila"
coords[None, :, :]   ->  (1, N, 3)     "cada átomo como columna"
                         ---------
al restarlos:            (N, N, 3)     el 1 se estira contra el N en ambos lados
```

El elemento `[i, j]` de ese resultado es el vector que va del átomo $j$ al átomo $i$. Luego solo
queda elevar al cuadrado, sumar las 3 componentes y hacer la raíz.

In [7]:
# una molécula de mentira: 4 átomos con coordenadas en angstroms
coords = np.array([[0.0, 0.0, 0.0],
                   [0.0, 0.0, 1.1],
                   [0.0, 1.0, 0.0],
                   [1.5, 0.0, 0.0]])

print("coords.shape          ", coords.shape)
print("coords[:, None, :].shape", coords[:, None, :].shape)
print("coords[None, :, :].shape", coords[None, :, :].shape)

diferencias = coords[:, None, :] - coords[None, :, :]
print("diferencias.shape     ", diferencias.shape, "  <- (N, N, 3)")

D = np.sqrt(np.sum(diferencias ** 2, axis=-1))    # axis=-1 = el último eje, el de las 3 coordenadas
print("\nmatriz de distancias (Å):\n", D)

# lo mismo, con einsum: 'ijk,ijk->ij' suma sobre k (las 3 coordenadas)
D_einsum = np.sqrt(np.einsum("ijk,ijk->ij", diferencias, diferencias))
print("\n¿coincide con einsum?", np.allclose(D, D_einsum))

coords.shape           (4, 3)
coords[:, None, :].shape (4, 1, 3)
coords[None, :, :].shape (1, 4, 3)
diferencias.shape      (4, 4, 3)   <- (N, N, 3)

matriz de distancias (Å):
 [[0.    1.1   1.    1.5  ]
 [1.1   0.    1.487 1.86 ]
 [1.    1.487 0.    1.803]
 [1.5   1.86  1.803 0.   ]]

¿coincide con einsum? True


La matriz sale como debe: **diagonal de ceros** (la distancia de un átomo a sí mismo) y
**simétrica** ($d_{ij} = d_{ji}$). Los valores cuadran con las coordenadas que pusimos: 1.1 y 1.0
para los átomos que están sobre un eje, 1.5 para el más alejado, y $\sqrt{1.1^2 + 1.0^2} = 1.487$
para los dos que están en ejes distintos.

Dos detalles que conviene fichar:

- **`axis=-1`** significa "el último eje", sea cual sea el rank. Es mucho más robusto que escribir
  `axis=2`: si mañana metes un eje de batch delante y pasas a `(32, N, N, 3)`, el `axis=-1` sigue
  siendo correcto y el `axis=2` ya no.
- **El coste.** Esto crea un array de $N \times N \times 3$. Para 4 átomos son 48 números; para una
  proteína de 10 000 átomos son 300 millones, y ahí ya hace falta otra estrategia (listas de vecinos
  con radio de corte). El broadcasting es elegante, pero **materializa** el array intermedio.

---

## 1.4 Cambiar el rank

Como el broadcasting alinea por el final, la mitad del trabajo con tensores es **colocar ejes de
tamaño 1 en el sitio adecuado**. Estas son las herramientas.

### `np.newaxis` (o `None`): añadir un eje

`np.newaxis` es literalmente un alias de `None`. Verás las dos formas; significan lo mismo.

| Expresión | De | A |
|---|---|---|
| `x[:, np.newaxis]` | `(6,)` | `(6, 1)` — columna |
| `x[np.newaxis, :]` | `(6,)` | `(1, 6)` — fila |
| `x[:, None, :]` | `(4, 3)` | `(4, 1, 3)` |

### `np.squeeze`: quitar los ejes de tamaño 1

Lo contrario. `np.squeeze` elimina **todos** los ejes que midan 1. Útil cuando una operación te
deja un `(1, 4, 1)` y querías un `(4,)`.

### `np.reshape`: reorganizar los números

Cambia la forma manteniendo el orden y la cantidad de números. La única regla es que el producto de
la forma nueva tenga que dar el mismo total.

El truco de `-1`: pones `-1` en **una** dimensión y numpy calcula sola cuánto vale.
`x.reshape(-1, 2)` significa "dos columnas, y las filas que hagan falta". Se usa constantemente
para dividir en batches: `datos.reshape(-1, batch_size, n_features)`.

### El ellipsis `...`: "todos los ejes que haya en medio"

Sirve para escribir código que funcione con tensores de rank desconocido. `T[..., 0]` es
"el índice 0 del último eje, sea cual sea el rank". Si mañana llega un eje de batch, sigue valiendo.

In [8]:
x = np.arange(6)
print("x                  ", x, " shape", x.shape)
print("x[:, np.newaxis]   shape", x[:, np.newaxis].shape, "  <- columna")
print("x[np.newaxis, :]   shape", x[np.newaxis, :].shape, "  <- fila")
print()
print("x.reshape(2, 3)    ->\n", x.reshape(2, 3))
print("\nx.reshape(-1, 2)   shape", x.reshape(-1, 2).shape, "  <- '-1' = 'calcula tú'")
print()
print("np.squeeze(np.ones((1,4,1))).shape ->", np.squeeze(np.ones((1, 4, 1))).shape)
print()
T = np.ones((5, 7, 3))
print("T.shape       ", T.shape)
print("T[..., 0].shape ", T[..., 0].shape, "  <- el primer valor del último eje")
print("T[0, ...].shape ", T[0, ...].shape, "  <- el primer elemento del primer eje")

x                   [0 1 2 3 4 5]  shape (6,)
x[:, np.newaxis]   shape (6, 1)   <- columna
x[np.newaxis, :]   shape (1, 6)   <- fila

x.reshape(2, 3)    ->
 [[0 1 2]
 [3 4 5]]

x.reshape(-1, 2)   shape (3, 2)   <- '-1' = 'calcula tú'

np.squeeze(np.ones((1,4,1))).shape -> (4,)

T.shape        (5, 7, 3)
T[..., 0].shape  (5, 7)   <- el primer valor del último eje
T[0, ...].shape  (7, 3)   <- el primer elemento del primer eje


---

## 1.5 Vista o copia

Un detalle de numpy que causa bugs difíciles de encontrar.

Cuando haces un **slice** (`a[:3]`) o un **reshape**, numpy normalmente **no copia** los datos: te
devuelve una **vista**, otra ventana sobre la misma memoria. Es una decisión de eficiencia — copiar
un array de un millón de números para mirar tres sería absurdo.

La consecuencia: **si modificas la vista, modificas el original.**

In [9]:
a = np.arange(6)
print("a original     ", a)

b = a[:3]          # slice -> VISTA
b[0] = 99
print("tras b[0] = 99 ", a, "  <- ¡ha cambiado 'a'!")
print("¿b es una vista de a?", b.base is a)

c = a.reshape(2, 3)   # reshape -> VISTA
c[0, 0] = -1
print("tras c[0,0]=-1 ", a, "  <- también")

d = a[[0, 1, 2]]      # fancy indexing (con lista) -> COPIA
d[0] = 0
print("tras d[0] = 0  ", a, "  <- este NO la cambia: era una copia")

a original      [0 1 2 3 4 5]
tras b[0] = 99  [99  1  2  3  4  5]   <- ¡ha cambiado 'a'!
¿b es una vista de a? True
tras c[0,0]=-1  [-1  1  2  3  4  5]   <- también
tras d[0] = 0   [-1  1  2  3  4  5]   <- este NO la cambia: era una copia


La regla práctica:

| Operación | ¿Vista o copia? |
|---|---|
| slice con `:` (`a[2:5]`, `a[:, 0]`) | **vista** |
| `reshape`, `transpose`, `ravel` | **vista** (casi siempre) |
| indexar con lista o array (`a[[0,2,4]]`) | **copia** |
| indexar con booleanos (`a[a > 0]`) | **copia** |
| `a.copy()`, `np.array(a)` | **copia**, garantizada |

Si vas a modificar algo y no quieres tocar el original: **`.copy()` explícito**. Cuesta nada
escribirlo y te ahorra una tarde de depuración.

> **Nota para más adelante:** en JAX (el capítulo 2 en adelante) este problema **no existe**,
> porque los arrays de JAX son **inmutables**: no se pueden modificar en el sitio. Por eso allí se
> escribe `w = w - eta * grad` y nunca `w -= eta * grad`. Es una restricción que JAX impone a
> propósito, precisamente para que la diferenciación automática pueda funcionar.

---

## 1.6 Resumen del capítulo 1

| Concepto | Lo esencial |
|---|---|
| **rank / shape** | cuántas dimensiones (`ndim`) y cuánto mide cada una (`shape`) |
| **primer eje = batch** | lee las formas en voz alta: `(32, 50, 3)` = 32 moléculas × 50 átomos × 3 coords |
| **einsum** | los índices que no aparecen en la salida se suman; es la notación de los papers |
| **element-wise** | `+ - * /` no cambian la forma; `*` NO es producto matricial (ese es `@`) |
| **reducciones** | `axis=k` significa "el eje k desaparece"; `keepdims=True` lo deja como 1 |
| **broadcasting** | se alinea **por el final**; los ejes deben ser iguales, o valer 1, o no existir |
| **`np.newaxis`** | añade un eje de tamaño 1 para que el broadcasting alinee donde tú quieres |
| **`reshape(-1, k)`** | `-1` = "calcula tú esta dimensión" |
| **`...`** | "todos los ejes de en medio", para código independiente del rank |
| **vista vs copia** | slices y reshapes comparten memoria; usa `.copy()` si vas a modificar |

**El hábito que hay que coger:** cuando algo no funcione, lo primero es **imprimir las formas**.

```python
print(a.shape, b.shape)
```

Nueve de cada diez bugs con tensores se resuelven ahí, antes de mirar los números.

**Patrón a recordar** (aparece en todos los capítulos de moléculas):

```python
diferencias = coords[:, None, :] - coords[None, :, :]      # (N, N, 3)
D = np.sqrt(np.sum(diferencias ** 2, axis=-1))             # (N, N)
```

---

**Siguiente:** capítulo 2, `02_introduccion.ipynb` — el primer modelo de ML de verdad.